<a href="https://colab.research.google.com/github/lsgrep/agents/blob/main/notebooks/02_context_economics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Context economics

**The claim you should be able to make when you finish:** *"Agent cost is
quadratic in turns, not linear. Caching divides the quadratic by ten; only
bounding the context changes its shape. And I can price a run before I build
it."*

Lab 1 ended on one sentence: **the transcript is the state, and everything in it
is billed again on every turn.** This lab works out what that costs, and the
answer surprises almost everyone the first time.

The shape of the argument is deliberately the same one a serving engineer uses
for a KV cache: write the closed form, predict, then measure, then explain the
gap. Thirty minutes, no API key.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "main"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 1. Predict first

Before running anything: an agent with a 1,200-token system prompt, 30 tools
(~6,000 tokens of schemas), a 300-token task, and per turn about 250 tokens of
assistant output and 1,400 tokens of tool result.

**Write down your guess for the total input tokens billed across a 40-turn run.**
Most people guess something near `40 × 7,500 = 300,000`.

Now derive it. On turn `i` the request contains everything so far:

```
turn 1:  P0
turn 2:  P0 + g
turn 3:  P0 + 2g
...
turn n:  P0 + (n-1)g
```

Sum that and you get

```
total_input = n*P0 + g * n(n-1)/2
```

Linear in the fixed prefix. **Quadratic in the tail.**

In [ ]:
from agentlab.budget import LoopShape, derive_loop_tokens

shape = LoopShape(
    system_tokens=1_200,     # your system prompt
    tool_tokens=6_000,       # 30 tool schemas, sent on EVERY request
    prompt_tokens=300,
    assistant_tokens=250,    # per turn: thinking + text + the tool_use block
    result_tokens=1_400,     # per turn: what the tool handed back
    turns=40,
)

print(derive_loop_tokens(shape))

1.59 million input tokens, not 300,000. **81% of the bill is resent transcript.**

The practical consequence is that run length is the dominant cost variable, and
the *tail* of your run-length distribution dominates your bill. A p99 run of 120
turns costs nine times a 40-turn run, not three times.

In [ ]:
from agentlab.budget import loop_input_tokens, peak_context

print(f"{'turns':>6} {'input tokens':>14} {'vs 20 turns':>12} {'peak context':>13}")
for n in (10, 20, 40, 80, 160):
    s = LoopShape(1_200, 6_000, 300, 250, 1_400, turns=n)
    total = loop_input_tokens(s)
    print(f"{n:>6} {total:>14,} {total / loop_input_tokens(LoopShape(1_200, 6_000, 300, 250, 1_400, turns=20)):>11.1f}x {peak_context(s):>13,}")

Note the two columns are answering different questions, and they fail
differently:

- **total input tokens** is the *bill*. It arrives at the end of the month.
- **peak context** is the *wall*. It arrives mid-run, as a hard error.

A run can be cheap and hit the wall, or fit comfortably and cost a fortune.
Track both.

## 2. Caching, and what it actually does

Prompt caching is the single highest-leverage change available. It is also the
one most often described half-correctly, as "caching makes it linear". It does
not.

The mechanism: turn `i` reads the transcript as of turn `i-1` at the cache-read
rate (0.1x input) and writes only the new delta (1.25x input for a 5-minute
TTL). The **token counts are identical**. Only the rate changes — and the rate
that got cheap is the one on the quadratic term.

In [ ]:
from agentlab.budget import derive_cache_savings

print(derive_cache_savings(shape))

82% off. Genuinely enormous, and worth doing before anything else on this page.

But look at what happens to the *shape* as runs get longer. If caching made cost
linear, doubling the turns would double the cost — a ratio of 2.0.

In [ ]:
from agentlab.budget import run_cost

print(f"{'turns':>6} {'uncached $':>11} {'cached $':>10} {'saving':>8} {'cached doubling':>16}")
prev = None
for n in (20, 40, 80, 160, 320):
    s = LoopShape(1_200, 6_000, 300, 250, 1_400, turns=n)
    cold, warm = run_cost(s, cached=False), run_cost(s, cached=True)
    ratio = f"{warm.usd / prev:.2f}x" if prev else "—"
    print(f"{n:>6} {cold.usd:>11,.2f} {warm.usd:>10,.2f} {1 - warm.usd / cold.usd:>7.0%} {ratio:>16}")
    prev = warm.usd

The cached doubling ratio starts around **2.5x** and climbs toward **3.5x**. It
is flattened at moderate lengths — because the cache *write* and the *output*
terms are both linear and they dilute the curve — and then the quadratic
reasserts itself as the run grows.

So the honest sentence is: **caching changes the slope, not the shape.**

### The thing that silently costs you all of it

The cache is a *prefix* match. Any byte that changes anywhere in the prefix
invalidates everything after it. The classic ways to lose the whole saving while
your code looks correct:

- a timestamp or a request id in the system prompt
- a tool list built from an unsorted dict, so the order wobbles
- adding or removing a tool mid-session
- switching model or effort mid-session
- **a stall longer than the TTL** — a slow tool, a human approval, a queue

The last one is not a bug you can grep for, which is why hit rate is a thing you
*measure*, never assume. `usage.cache_read_input_tokens` is the only proof.

In [ ]:
print(f"{'hit rate':>9} {'cost $':>9} {'vs perfect':>11}")
perfect = run_cost(shape, cached=True, cache_hit_rate=1.0).usd
for hit in (1.0, 0.9, 0.7, 0.5, 0.0):
    usd = run_cost(shape, cached=True, cache_hit_rate=hit).usd
    print(f"{hit:>9.0%} {usd:>9,.2f} {usd / perfect:>10.1f}x")

print("\nA 50% hit rate is not half the saving. It is roughly five times the cost.")

## 3. The lever that changes the shape

Caching multiplies the quadratic by 0.1. To get rid of the quadratic you have to
stop the transcript from growing — compact it, clear old results, or keep data
behind handles. Lab 7 is about what each of those *loses*. This section is only
about what they cost.

Predict again before running: for our shape, at what run length does compacting
at 60K tokens start to be cheaper than not compacting?

In [ ]:
from agentlab.budget import bounded_run_cost

print(f"{'turns':>6} {'peak':>9} {'unbounded $':>12} {'bounded $':>10}  verdict")
for n in (25, 35, 40, 45, 60, 100, 200):
    s = LoopShape(1_200, 6_000, 300, 250, 1_400, turns=n)
    unbounded, bounded = run_cost(s, cached=True), bounded_run_cost(s, cap=60_000)
    verdict = "bounding pays" if bounded.usd < unbounded.usd else "bounding COSTS you"
    print(f"{n:>6} {peak_context(s):>9,} {unbounded.usd:>12,.2f} {bounded.usd:>10,.2f}  {verdict}")

**Bounding a run that was never long enough to need it costs more than it
saves.** The compaction fires once, near the end, rewrites the whole prefix into
a fresh cache entry, and buys you two turns of slightly cheaper reads.

This is a real and common way to ship a context strategy that loses money, and
it is invisible unless you look — the feature *works*, it just costs more.
Crossover here is somewhere past 40 turns. Find yours.

At 200 turns the picture is completely different, because that is where the
quadratic has properly taken over.

## 4. `P0`: the term everyone optimises first

The fixed prefix is the visible one, so it gets the attention. It is also
usually the smallest of the three levers — but it is not nothing, and tool
schemas are the part people forget entirely, because they do not appear in any
prompt file.

In [ ]:
from agentlab.budget import context_share, tool_overhead

print(f"{'tools':>6} {'tokens/request':>15} {'% of 1M window':>15} {'over 40 turns':>15}")
for n in (5, 15, 30, 60, 150):
    overhead = tool_overhead(n)
    print(f"{n:>6} {overhead:>15,} {context_share(overhead):>14.2%} {overhead * 40:>15,}")

print("\n150 tools is ~57K tokens of every single request, and 2.3M across a 40-turn run —")
print("before the agent does anything. Lab 4 measures what it does to accuracy, too.")

## 5. Your own run

The point of all this is to run it on *your* numbers. `Trace.shape()` fits a
`LoopShape` to a real trajectory, so you can price a run you have already done
and project one you have not.

In [ ]:
from agentlab.loop import ModelResponse, PolicyModel, Tool, ToolRegistry, run, text_block, tool_use_block

DOCS = {f"doc-{i}": "lorem ipsum " * 200 for i in range(30)}
tools = ToolRegistry([
    Tool("read_doc", "Read one document in full and return its text.",
         {"type": "object", "properties": {"doc_id": {"type": "string"}}, "required": ["doc_id"]},
         fn=lambda doc_id: DOCS.get(doc_id, "not found")),
])

def reader(messages, tools_):
    read = sum(1 for m in messages if m["role"] == "assistant"
               for b in m["content"] if b.get("type") == "tool_use")
    if read < 12:
        return ModelResponse([tool_use_block(f"t{read}", "read_doc", {"doc_id": f"doc-{read}"})], "tool_use")
    return ModelResponse([text_block("Summary of 12 documents.")], "end_turn")

trace = run(PolicyModel(reader), tools, "Summarise the corpus.", max_steps=20)
fitted = trace.shape()
print(trace)
print(f"\nfitted shape: {fitted}")
print(f"\n{'turns':>6} {'cached $':>10}")
for n in (fitted.turns, 40, 100):
    print(f"{n:>6} {run_cost(fitted.at_turns(n), cached=True).usd:>10,.4f}")
print("\nSame agent, same tools. Only the run length changed.")

## 6. The whole chain at once

`budget.worksheet()` runs the full sizing pass — tokens, caching, and a bound —
with every substitution shown. This is the thing to put in front of someone who
asks "what will this cost".

In [ ]:
from agentlab.budget import worksheet

print(worksheet(shape, cap=60_000))

## A note on the prices

`budget.PRICES` is a **snapshot you maintain**, stamped with `VERIFIED_ON`. API
pricing moves, intro rates expire, and a lab that quotes a stale number
confidently is worse than one that quotes none.

In [ ]:
from agentlab.budget import PRICES, VERIFIED_ON, staleness

print(f"snapshot verified {VERIFIED_ON} — {staleness()} days ago")
for name, p in PRICES.items():
    print(f"  {name:<20} in ${p.input:>5.2f}  out ${p.output:>6.2f}  "
          f"cache read ${p.cache_read:>4.2f}  write ${p.cache_write():>5.2f}  ctx {p.context:,}")
print("\nRe-check before quoting anything to anyone. staleness() warns; it cannot verify.")

## What you can now say

- *"Agent cost is quadratic in turns. Forty turns isn't twice twenty, it's about
  four times, and 81% of the bill is resent transcript."*
- *"Caching divides the quadratic by ten — it changes the slope, not the shape.
  The cached doubling ratio still climbs from 2.5x toward 3.5x."*
- *"A 50% cache hit rate isn't half the saving, it's five times the cost — so we
  measure it from `cache_read_input_tokens` rather than assuming it."*
- *"Total tokens is the bill; peak context is the wall. Different numbers."*
- *"Compacting a run that never needed it costs more than it saves. Our
  crossover is at N turns."*

## Next

You can now price a run. **[Lab 3](03_reliability_math.ipynb) asks whether it will
finish** — and the arithmetic there is even less forgiving.